## demonstrate that we have a project

In [ ]:
import numpy as np
import pylab as plt
from astropy.timeseries import LombScargle

In [ ]:
rng = np.random.default_rng(17) # the most random number

In [ ]:
# make *extremely unrealistic* toy fake data
N = 128
times = np.sort(rng.uniform(size=N)) * 5. * 365.25 # 5 years in days
period = 1.3 * np.pi * 365.25 # 1.5 * pi years in days
omega = 2. * np.pi / period
phi_0 = 5.2331 # radians
scan_angles = rng.uniform(size=N) * 2. * np.pi # one circle in radians
a = 2.155 # arbitrary units
psi_0 = 1.1 # radians
b = 1.344 # arbitrary units; same as a
thetas = np.array([a * np.cos(psi_0), a * np.sin(psi_0)])[None, :] * np.cos(omega * times[:, None] - phi_0) \
       + np.array([-b * np.sin(psi_0), b * np.cos(psi_0)])[None, :] * np.sin(omega * times[:, None] - phi_0)
ivars = 1.0 + 1.5 * rng.uniform(size=N)
errs = 1. / np.sqrt(ivars)
noises = rng.normal(size=N) * errs
ys = np.sum(thetas * np.vstack([np.cos(scan_angles - psi_0), np.sin(scan_angles - psi_0)]).T, axis=1)

In [ ]:
# check the (noiseless) internals of the generative model
plt.figure(figsize=(4, 4))
plt.scatter(thetas[:, 0], thetas[:, 1], c=times)
plt.axis("equal")

In [ ]:
# check that the data look "okay"
plt.figure(figsize=(8, 4))
plt.errorbar(times, ys, marker="none", color="k", linestyle="none", yerr=errs)
plt.scatter(times, ys, c=scan_angles)

In [ ]:
# take the standard periodogram
fs = np.arange(1. / 3650., 1. / 30., 1. / (5. * 3650.)) # over-sampling
powers = LombScargle(times, ys, errs, normalization="psd").power(fs)

In [ ]:
# nothing to see here
plt.figure(figsize=(8, 4))
plt.axvline(period, c="g")
plt.plot(1. / fs, powers, "k-")

In [ ]:
# now hack the LSP
# Note that what we are doing is: weighting the data by the cosines and sines of the scan angles.
# And: we are re-weighting the inverse variances appropriately.
powers0 = LombScargle(times, ys * np.cos(scan_angles),
             1. / np.sqrt(ivars * np.cos(scan_angles) ** 2), normalization="psd").power(fs)
powers1 = LombScargle(times, ys * np.sin(scan_angles),
             1. / np.sqrt(ivars * np.sin(scan_angles) ** 2), normalization="psd").power(fs)

In [ ]:
# we have a signal!
plt.figure(figsize=(8, 4))
plt.axvline(period, c="g")
plt.plot(1. / fs, powers0, "r-")
plt.plot(1. / fs, powers1, "b-")
plt.plot(1. / fs, powers0 + powers1, "k-")